# Семинар 1: Практическое введение в PyTorch и обучение первой нейронной сети

### Введение: Что такое PyTorch и как устроено обучение нейросетей?
**PyTorch** — это библиотека для языка Python, которая стала стандартом де-факто в индустрии искусственного интеллекта и научных исследованиях. Если вы уже знакомы с библиотекой `NumPy` для работы с массивами чисел, освоить PyTorch будет очень легко.

**Зачем вообще нужен специальный фреймворк, если есть `NumPy`?**
Обучение нейросети — это миллионы однотипных математических операций (умножений и сложений матриц) и поиск минимума сложной функции с помощью производных. PyTorch решает две фундаментальные проблемы:
1. **Ускорение вычислений на GPU (видеокарте):** Обычный процессор (CPU) имеет несколько мощных ядер и решает задачи последовательно. Видеокарта (GPU) имеет тысячи простых ядер и умеет перемножать огромные матрицы одновременно. PyTorch позволяет перенести расчеты на видеокарту одной строчкой кода.
2. **Автоматическое дифференцирование (`autograd`):** Чтобы нейросеть училась, нам нужно знать, в какую сторону и насколько сильно изменить каждый из миллионов её весов, чтобы уменьшить итоговую ошибку. Вручную выводить производные для глубоких сетей практически невозможно. PyTorch запоминает все математические действия, которые мы совершаем над данными, и сам вычисляет нужные производные.

---
### Часть 0. Подключение GPU (видеокарты) и управление устройствами (`device`)
В PyTorch любой тензор и любая модель живут на конкретном вычислительном устройстве: либо в оперативной памяти обычного процессора (`cpu`), либо в видеопамяти видеокарты (`cuda` для NVIDIA или `mps` для чипов Apple Silicon).

**Три золотых правила работы с GPU в PyTorch:**
1. **Универсальный код через переменную `device`:** Никогда не пишите жестко `.cuda()` в коде — если запустить такой ноутбук на компьютере без видеокарты, программа упадет с ошибкой. Правильно определить доступное устройство автоматически и использовать `.to(device)`.
2. **Всё должно лежать на одном устройстве:** Если ваша нейросеть перенесена на GPU, а входной батч данных остался на CPU, PyTorch выдаст ошибку `RuntimeError: Expected all tensors to be on the same device`. И модель, и данные нужно отправлять на `device`.
3. **Перед переводом в NumPy или Matplotlib нужно вернуть тензор на CPU:** Библиотеки `NumPy` и `matplotlib` не умеют читать память видеокарты. Поэтому перед отрисовкой графиков мы пишем `tensor.cpu().numpy()` (или `tensor.detach().cpu().numpy()`, если у тензора включено отслеживание градиентов).

In [ ]:
import torch
import time

# 1. Автоматически определяем лучшее доступное устройство
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU найден! Используем видеокарту: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("GPU найден! Используем ускоритель Apple Silicon (MPS)")
else:
    device = torch.device("cpu")
    print("GPU не найден. Вычисления будут проходить на обычном процессоре (CPU).")

# 2. Как перенести тензор на выбранное устройство?
t_cpu = torch.randn(3, 3)                  # По умолчанию создается на CPU
t_device = t_cpu.to(device)                # Переносим на GPU (если он есть)
print(f"Устройство t_cpu: {t_cpu.device} | Устройство t_device: {t_device.device}")

# 3. Бенчмарк: сравним скорость перемножения больших матриц (3000 x 3000) на CPU и GPU
size = 3000
A_cpu = torch.randn(size, size)
B_cpu = torch.randn(size, size)

start = time.time()
C_cpu = A_cpu @ B_cpu
cpu_time = time.time() - start
print(f"\nВремя умножения матриц {size}x{size} на CPU: {cpu_time:.4f} сек.")

if device.type != "cpu":
    A_gpu = A_cpu.to(device)
    B_gpu = B_cpu.to(device)
    # Прогрев GPU перед замером
    _ = A_gpu @ B_gpu
    if device.type == "cuda":
        torch.cuda.synchronize()
        
    start = time.time()
    C_gpu = A_gpu @ B_gpu
    if device.type == "cuda":
        torch.cuda.synchronize()  # Ждем реального завершения операций на видеокарте
    gpu_time = time.time() - start
    print(f"Время умножения матриц {size}x{size} на GPU: {gpu_time:.4f} сек. (Ускорение в {cpu_time / gpu_time:.1f} раз!)")

---
### Часть 1. Тензоры (`torch.Tensor`) — фундамент PyTorch
В мире глубокого обучения любые данные представляются в виде **тензоров**:
* **0D-тензор (скаляр):** одно число (например, значение ошибки `loss = 0.45`).
* **1D-тензор (вектор):** одномерный список чисел (например, признаки одного клиента банка: `[возраст, доход, стаж]`).
* **2D-тензор (матрица):** таблица чисел (например, таблица из 100 клиентов, где у каждого по 3 признака — форма `(100, 3)`).
* **3D- и 4D-тензоры:** цветные изображения (высота × ширина × 3 цветовых канала RGB) или видео.

Главное правило PyTorch: **все данные на вход нейросети должны подаваться в виде `torch.Tensor`**, а не списков Python или массивов `NumPy`.

In [ ]:
# Фиксируем зерно генератора случайных чисел (seed).
# Это гарантирует, что у всех участников семинара случайные числа сгенерируются одинаково.
torch.manual_seed(42)

# 1. Создание тензора из обычного списка Python
a = torch.tensor([[2.0, 4.0], 
                  [1.0, 3.0]])

# Создание тензора из единиц и умножение на число (получим матрицу 2x2 из троек)
b = torch.ones((2, 2)) * 3

# Генерация случайного тензора размером 2 строки на 3 столбца (из нормального распределения)
rand_t = torch.randn(2, 3)

print("Тензор a:\n", a)
print("Форма (размерность) тензора a:", a.shape)

# 2. Арифметические операции
# Знак '+' выполняет поэлементное сложение (каждый элемент складывается с соответствующим)
print("\nПоэлементное сложение (a + b):\n", a + b)

# Знак '@' выполняет классическое матричное умножение (строка на столбец)
# Важно: для матричного умножения (2x2) @ (2x3) внутренние размерности (2 и 2) должны совпадать!
print("\nМатричное умножение (a @ rand_t):\n", a @ rand_t)

# 3. Объединение (конкатенация) тензоров
c = torch.tensor([[5.0, 6.0]]) # Форма (1, 2)
# dim=0 означает склейку вдоль нулевой оси (добавляем новую строку снизу)
concat_rows = torch.cat([a, c], dim=0)
print("\nПосле добавления строки (dim=0):\n", concat_rows)
print("Новая форма:", concat_rows.shape)

# 4. Как достать обычное число из тензора обратно в Python?
# Метод .item() работает только для тензоров, состоящих ровно из одного числа!
single_tensor = torch.tensor([7.5])
print("\nОбычное число Python:", single_tensor.item(), "| Тип:", type(single_tensor.item()))

# Перевод обратно в NumPy (нужно, например, для построения графиков через matplotlib)
numpy_arr = a.numpy()
print("Тип после .numpy():", type(numpy_arr))

#### Мини-эксперимент №1: Понимание размерностей (`dim`)
Новички чаще всего допускают ошибки в PyTorch из-за несовпадения размерностей (ошибка `RuntimeError: Sizes of tensors must match`).
* **Задание:** В ячейке выше создайте тензор-столбец `d = torch.tensor([[5.0], [6.0]])` (его форма `2x1`) и попробуйте приклеить его к тензору `a` (форма `2x2`) справа с помощью `torch.cat([a, d], dim=1)`.
* **Подсказка и ответ:**
  * Ось `dim=0` — это движение **вниз по строкам** (вертикально). При склейке по `dim=0` должна совпадать ширина (число столбцов).
  * Ось `dim=1` — это движение **вправо по столбцам** (горизонтально). При склейке по `dim=1` должна совпадать высота (число строк).
  * После выполнения `torch.cat([a, d], dim=1)` получится матрица размером `(2, 3)`.

---
### Часть 2. Из чего состоит нейросеть? (`torch.nn`)
Искусственный нейрон устроен очень просто: он берет входные числа, умножает каждое на свой «вес» (коэффициент важности), складывает их и прибавляет число смещения (*bias*).
В PyTorch такой блок называется **полносвязным линейным слоем** — `nn.Linear(in_features, out_features)`:
* `in_features` — сколько чисел приходит на вход слою.
* `out_features` — сколько нейронов в этом слое (столько чисел выйдет наружу).

**Зачем нужны функции активации (`ReLU`, `Sigmoid`, `Tanh`)?**
Если поставить друг за другом два линейных слоя без функции активации, математически они схлопнутся в один линейный слой. Такая сеть сможет проводить только прямые линии и никогда не научится распознавать сложные образы. **Функция активации** пропускает выход нейрона через нелинейное правило (например, `ReLU` заменяет все отрицательные числа на ноль, а положительные оставляет без изменений).

In [ ]:
import torch.nn as nn

# Создаем линейный слой: принимает 4 признака, на выходе выдает 2 числа
layer = nn.Linear(in_features=4, out_features=2, bias=True)
activation = nn.ReLU()

# Имитируем входные данные: 1 объект с 4 признаками
sample_input = torch.tensor([1.5, -2.0, 0.5, 3.0])

# Пропускаем данные через слой, а затем через функцию активации
linear_out = layer(sample_input)
activated_out = activation(linear_out)

print("1. Случайные начальные веса слоя W (форма 2x4):\n", layer.weight)
print("\n2. Смещения слоя b (форма 2):\n", layer.bias)
print("\n3. Выход до активации:", linear_out)
print("4. Выход после ReLU (отрицательные числа обнулились):", activated_out)

> **Важное наблюдение для новичков:** Посмотрите на конец вывода в пункте 4 — там написано `grad_fn=<ReluBackward0>`. Когда вы создаете слой `nn.Linear`, PyTorch автоматически помечает его веса как обучаемые параметры (`requires_grad=True`). При каждом вычислении с этими весами PyTorch строит невидимый граф операций, чтобы потом можно было вызвать `.backward()` и узнать, как изменить веса.

---
### Часть 3. Практическая задача: Классификация «Двух лун»
Решим классическую задачу **бинарной классификации**: у нас есть точки на двумерной плоскости (у каждой точки две координаты: $X_1$ и $X_2$), и каждая точка принадлежит либо к **Классу 0** (красные), либо к **Классу 1** (синие).

Мы специально выбрали датасет **«Две луны» (`make_moons`)**, где два класса изогнуты в виде полумесяцев, вложенных друг в друга. Обычная линейная модель (логистическая регрессия) способна провести только одну прямую линию и неизбежно ошибется на концах полумесяцев. Посмотрим, как с этим справится двухслойная нейросеть!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# Генерируем 1000 точек. Параметр noise добавляет случайный разброс, как в реальных данных
X, y = make_moons(n_samples=1000, noise=0.18, random_state=42)

# Обязательный шаг в машинном обучении: делим данные на две части.
# На X_train (800 точек) сеть будет учиться, а на X_test (200 точек) мы честно проверим её качество.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Размер обучающей выборки: {X_train.shape}, тестовой: {X_test.shape}")

# Отрисовываем обучающие точки
plt.figure(figsize=(8, 5))
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], color='crimson', label='Класс 0', alpha=0.7)
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], color='royalblue', label='Класс 1', alpha=0.7)
plt.title("Обучающие данные: Две луны (make_moons)")
plt.xlabel("Координата X1")
plt.ylabel("Координата X2")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

#### Сборка архитектуры нейросети в PyTorch
В PyTorch любую архитектуру принято оформлять в виде класса, унаследованного от `nn.Module`. Внутри него всегда два главных метода:
1. `__init__(self)` — здесь мы «покупаем детали конструктора» (создаем слои и функции активации).
2. `forward(self, x)` — здесь мы «соединяем детали» (описываем, в каком порядке данные `x` проходят через слои).

**Как устроена наша сеть `MoonClassifier`?**
* **Вход:** 2 числа (координаты точки $x_1$ и $x_2$).
* **Скрытый слой:** `nn.Linear(2, 16)` + `nn.ReLU()`. 16 нейронов в скрытом слое позволят сети «нарисовать» сложную ломаную границу вокруг полумесяцев.
* **Выходной слой:** `nn.Linear(16, 1)` + `nn.Sigmoid()`. На выходе нам нужно одно число от `0` до `1` — вероятность того, что точка относится к Классу 1. Именно функция `Sigmoid` превращает любое число в вероятность от 0 до 1.

Обратите внимание: при создании модели мы сразу переносим её на выбранное устройство с помощью `.to(device)`.

In [ ]:
class MoonClassifier(nn.Module):
    def __init__(self, hidden_neurons=16):
        super().__init__()
        # Объявляем слои
        self.fc1 = nn.Linear(2, hidden_neurons)
        self.act1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_neurons, 1)
        self.act2 = nn.Sigmoid()

    def forward(self, x):
        # Описываем маршрут данных (прямой проход)
        x = self.fc1(x)
        x = self.act1(x)
        x = self.fc2(x)
        x = self.act2(x)
        return x

# Создаем экземпляр модели и переносим её веса на device (GPU или CPU)
model = MoonClassifier(hidden_neurons=16).to(device)

# Проверим, как работает сеть ДО обучения (сейчас её веса заполнены случайным шумом).
# Блок 'with torch.no_grad():' говорит PyTorch: "Мы сейчас только проверяем модель,
# учить её не нужно, поэтому не трать память на построение графа градиентов".
with torch.no_grad():
    # Переносим тестовые данные на то же устройство, где лежит модель!
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    probabilities = model(X_test_tensor)
    
    # Перед переводом в NumPy обязательно возвращаем тензор обратно на CPU (.cpu())
    predictions = (probabilities.cpu().numpy().ravel() > 0.5).astype(int)
    accuracy = (predictions == y_test).mean()

print(f"Точность необученной нейросети на тесте: {accuracy:.2%} (почти как подбрасывание монетки)")

---
### Часть 4. Как происходит обучение? (Цикл обучения и `DataLoader`)
Обучение нейросети — это многократное повторение одних и тех же шагов. Разберем ключевые термины, которые встретятся в коде:
* **Эпоха (Epoch):** один полный проход нейросети через всю обучающую выборку (все 800 точек).
* **Батч (Batch) и `DataLoader`:** если в датасете миллион картинок, подавать их в сеть все сразу невозможно — не хватит памяти видеокарты. Поэтому данные нарезают на маленькие порции — **батчи** (например, по 32 точки). Класс `DataLoader` сам нарезает данные на батчи и перемешивает их перед каждой эпохой (`shuffle=True`), чтобы сеть не зазубривала порядок примеров.
* **Функция потерь (`nn.BCELoss` — Binary Cross-Entropy):** «судья», который сравнивает предсказанные сетью вероятности с правильными ответами и выдает одно число — величину ошибки (`loss`). Чем меньше `loss`, тем лучше работает сеть.
* **Оптимизатор (`torch.optim.Adam`):** «тренер», который смотрит на градиенты ошибки и решает, как подкрутить веса каждого нейрона. Параметр `lr` (*learning rate*, скорость обучения) определяет размер шага при обновлении весов.

**Золотое правило 4 шагов внутри цикла обучения:**
1. `loss = criterion(y_pred, y_batch)` — считаем ошибку.
2. `optimizer.zero_grad()` — **обнуляем старые градиенты** (иначе PyTorch прибавит новые градиенты к градиентам с прошлого шага, и обучение сломается).
3. `loss.backward()` — **считаем новые градиенты** (алгоритм обратного распространения ошибки).
4. `optimizer.step()` — **обновляем веса** модели.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# 1. Подготовка данных в формате тензоров PyTorch
X_train_t = torch.FloatTensor(X_train)
# Метод .unsqueeze(1) превращает вектор формы (800,) в столбец формы (800, 1),
# чтобы его форма совпадала с выходом нейросети.
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)

# Упаковываем признаки и ответы в Dataset и создаем DataLoader
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 2. Инициализация модели на выбранном устройстве (device), функции потерь и оптимизатора
torch.manual_seed(42)
model = MoonClassifier(hidden_neurons=16).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

epochs = 80
loss_history = []  # Сюда будем записывать среднюю ошибку на каждой эпохе

# 3. Главный цикл обучения
for epoch in range(epochs):
    batch_losses = []
    
    # Проходим по всем мини-батчам в обучающей выборке
    for X_batch, y_batch in train_loader:
        # Переносим очередной батч данных на GPU/CPU
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        # Прямой проход: получаем предсказания и считаем ошибку
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        
        # Обратный проход и шаг оптимизации
        optimizer.zero_grad()  # Шаг 1: очищаем старые градиенты
        loss.backward()        # Шаг 2: считаем новые градиенты
        optimizer.step()       # Шаг 3: обновляем веса
        
        batch_losses.append(loss.item())
        
    # Сохраняем среднюю ошибку за эпоху
    loss_history.append(np.mean(batch_losses))

# Визуализируем процесс обучения
plt.figure(figsize=(8, 4))
plt.plot(loss_history, color='darkorange', linewidth=2)
plt.title(f"График падения ошибки (Loss) во время обучения [Устройство: {device}]")
plt.xlabel("Номер эпохи")
plt.ylabel("Ошибка (Binary Cross-Entropy)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
from matplotlib.colors import ListedColormap

# Считаем итоговую точность на отложенной тестовой выборке
with torch.no_grad():
    test_preds = model(torch.FloatTensor(X_test).to(device)).cpu().numpy().ravel()
    test_acc = ((test_preds > 0.5).astype(int) == y_test).mean()

print(f"Точность ОБУЧЕННОЙ нейросети на тесте: {test_acc:.2%}")

# Создаем плотную сетку из 40 000 точек по всей плоскости для отрисовки фона
plt.figure(figsize=(10, 6))
eps = 0.2
xx, yy = np.meshgrid(
    np.linspace(X_test[:, 0].min() - eps, X_test[:, 0].max() + eps, 200),
    np.linspace(X_test[:, 1].min() - eps, X_test[:, 1].max() + eps, 200)
)

with torch.no_grad():
    grid_tensor = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()]).to(device)
    Z = (model(grid_tensor).cpu().numpy() > 0.5).astype(int).reshape(xx.shape)

# Закрашиваем области решений и наносим поверх тестовые точки
plt.pcolormesh(xx, yy, Z, cmap=ListedColormap(['#FFCCCC', '#CCE5FF']), shading='auto')
plt.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1], color='crimson', edgecolor='k', label='Класс 0 (тест)')
plt.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1], color='royalblue', edgecolor='k', label='Класс 1 (тест)')
plt.title(f"Разделяющая граница нейросети (Точность на тесте: {test_acc:.1%})")
plt.xlabel("Координата X1")
plt.ylabel("Координата X2")
plt.legend()
plt.show()

#### Мини-эксперимент №2: «Поменяй и посмотри» (с разбором для семинара)
Лучший способ понять внутреннюю работу нейросети — намеренно «сломать» или изменить её настройки и посмотреть на график. Проведите эти 4 эксперимента по очереди, меняя код в ячейках выше:

1. **Эксперимент А: Что будет, если убрать функцию активации `ReLU`?**
   * *Что сделать:* В методе `forward` класса `MoonClassifier` закомментируйте строку `x = self.act1(x)` и перезапустите ячейки с обучением и графиком.
   * *Что вы увидите:* Разделяющая граница превратится в **идеально прямую линию**, а точность упадет с ~97% до ~84%.
   * *Пояснение:* Без нелинейной функции активации два последовательных линейных слоя математически эквивалентны одному линейному слою. Сеть потеряла способность изгибать границу и превратилась в обычную линейную классификацию.

2. **Эксперимент Б: Сравнение классического `SGD` и современного `Adam`**
   * *Что сделать:* Верните `ReLU`, а в ячейке обучения замените оптимизатор `torch.optim.Adam(model.parameters(), lr=0.02)` на базовый стохастический градиентный спуск `torch.optim.SGD(model.parameters(), lr=0.02)`.
   * *Что вы увидите:* За те же 80 эпох ошибка уменьшится гораздо слабее, а граница останется почти прямой (точность около 83–85%).
   * *Пояснение:* Обычный `SGD` обновляет все веса с одинаковым фиксированным шагом `lr`, поэтому медленно выбирается из пологих участков функции потерь. `Adam` хранит «инерцию» (импульс) предыдущих шагов и автоматически подбирает индивидуальную скорость обучения для каждого веса, поэтому сходится в разы быстрее.

3. **Эксперимент В: Слишком маленький скрытый слой (Недообучение / Underfitting)**
   * *Что сделать:* Верните `Adam`, но при создании модели укажите всего 2 нейрона в скрытом слое: `model = MoonClassifier(hidden_neurons=2).to(device)`.
   * *Что вы увидите:* Граница будет состоять максимум из двух прямых отрезков (одного «угла»).
   * *Пояснение:* Каждый нейрон со слоем `ReLU` способен создать один «излом» в пространстве. Двух нейронов физически недостаточно, чтобы плавно обогнуть оба полумесяца.

4. **Эксперимент Г: Слишком большой шаг обучения (`lr`)**
   * *Что сделать:* Верните `hidden_neurons=16`, но поставьте огромный шаг обучения `lr=2.0` в оптимизаторе `Adam`.
   * *Что вы увидите:* График ошибки (`Loss`) начнет хаотично прыгать вверх-вниз, а модель не сможет нормально обучиться.
   * *Пояснение:* Из-за слишком большого шага оптимизатор на каждой итерации «перепрыгивает» дно ямы (минимум функции потерь) и улетает на противоположный склон.

---
### Часть 5. Компактная запись через `nn.Sequential`
Когда ваша нейросеть представляет собой прямой конвейер (данные просто идут от первого слоя к последнему без сложных развилок), описывать целый класс с методом `forward` необязательно. В PyTorch для этого есть контейнер `nn.Sequential` — вы просто перечисляете слои друг за другом, как бусины на нитке.

In [ ]:
# Создаем точно такую же нейросеть, как MoonClassifier(hidden_neurons=16), и переносим на device
seq_model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
    nn.Sigmoid()
).to(device)

print("Архитектура модели:\n", seq_model)

# Проверим, что модель работает: подадим ей батч из 5 случайных точек на том же устройстве
sample_batch = torch.randn(5, 2, device=device)
with torch.no_grad():
    output_probs = seq_model(sample_batch)

print("\nПредсказанные вероятности для 5 точек:\n", output_probs)

> **Когда что использовать?** `nn.Sequential` идеален для простых последовательных блоков и быстрых прототипов. Но как только вам понадобится архитектура, где данные разветвляются на два потока или перепрыгивают через слои (как в современных сетях ResNet или трансформерах), вам понадобится гибкость полноценного класса `nn.Module` с собственным методом `forward`.